In [1]:
!pip install -q pyngrok

In [2]:
import subprocess

print("Installing zstd dependency...")
subprocess.run("apt-get update && apt-get install -y zstd", shell=True, check=True)

print("Installing Ollama engine...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

Installing zstd dependency...
Get:1 https://cli.github.com/packages stable InRelease [4,685 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,578 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [112 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,920 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.8 MB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:13 https://ppa.laun

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)



Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 220 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (4,515 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
Installing Ollama engine...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...


>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


CompletedProcess(args='curl -fsSL https://ollama.com/install.sh | sh', returncode=0)

In [3]:
import os
import time
import requests
import subprocess
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    NGROK_TOKEN = user_secrets.get_secret("NGROK_AUTH_TOKEN")
except Exception as e:
    print("❌ ERROR: Could not find 'NGROK_AUTH_TOKEN' in Kaggle Secrets.", flush=True)
    raise e

print("Opening ngrok tunnel...", flush=True)
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(11434, host_header="localhost").public_url

# Environment settings optimized for high-depth thinking & parallel queries
os.environ["OLLAMA_NUM_PARALLEL"] = "4"
os.environ["OLLAMA_MAX_QUEUE"] = "10" 
os.environ["OLLAMA_KEEP_ALIVE"] = "60m"
os.environ["OLLAMA_HOST"] = "0.0.0.0" 
os.environ["OLLAMA_CONTEXT_LENGTH"] = "65536"

print("\nStarting multi-model Ollama server...", flush=True)
server_process = subprocess.Popen("ollama serve", shell=True, env=os.environ)
time.sleep(5) 

print("-" * 65, flush=True)
print("Downloading models sequentially (stable & deadlock-free)...", flush=True)

# Pull primary model: Qwen 3.8 27B
print("\n📦 Pulling primary model: qwen3.8:27b...", flush=True)
subprocess.run("ollama pull qwen3.8:27b", shell=True, check=True)

# Pull auxiliary models
print("\n📦 Pulling model: qwen3.6:27b...", flush=True)
subprocess.run("ollama pull qwen3.6:27b", shell=True, check=True)

print("\n📦 Pulling model: qwen3.6:latest...", flush=True)
subprocess.run("ollama pull qwen3.6:latest", shell=True, check=True)

print("\n✅ All models downloaded and verified!", flush=True)

# Instant non-blocking GPU VRAM warmup (pre-loads weights with zero generation lag)
print("\n🔥 Warming up Qwen3.8 into GPU VRAM in xhigh thinking mode...", flush=True)
try:
    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen3.8:27b",
            "keep_alive": "60m",
            "stream": False
        },
        timeout=45
    )
    if res.status_code == 200:
        print("✅ Successfully pre-loaded qwen3.8:27b directly into GPU VRAM!", flush=True)
    else:
        print(f"⚠️ Warmup returned status {res.status_code}. Model will load on first query.", flush=True)
except Exception as e:
    print(f"⚠️ Warmup note: {e}", flush=True)

print("\n" + "="*65, flush=True)
print("✅ MULTI-MODEL OLLAMA SERVER IS ONLINE WITH QWEN 3.8 WARMED UP!")
print(f"🚀 Base URL for your coding agent:")
print(f"🔗 {public_url}/v1", flush=True)
print("="*65 + "\n", flush=True)

try:
    print("Server is active and ready for requests. Keep this browser tab open!", flush=True)
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nShutting down server and tunnel...", flush=True)
    server_process.terminate()
    ngrok.kill()


Opening ngrok tunnel...
                                                                                                    
Starting multi-model Ollama server...
Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIHYHjTYAy9yUd0yHKi1TzX2E7jXl/jr9jJ/UbiqJPPJn



time=2026-09-02T06:12:13.977Z level=INFO source=routes.go:1951 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:65536 OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://0.0.0.0:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:1h0m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:10 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:4 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0

-----------------------------------------------------------------
Initiating parallel downloads for all models...


time=2026-09-02T06:12:21.828Z level=INFO source=types.go:32 msg="inference compute" id=0 filter_id=0 library=CUDA compute=7.5 name=CUDA0 description="Tesla T4" libdirs=ollama,cuda_v13 driver=13.0 pci_id=0000:00:04.0 type=discrete total="14.6 GiB" available="14.5 GiB"
time=2026-09-02T06:12:21.828Z level=INFO source=types.go:32 msg="inference compute" id=1 filter_id=1 library=CUDA compute=7.5 name=CUDA1 description="Tesla T4" libdirs=ollama,cuda_v13 driver=13.0 pci_id=0000:00:05.0 type=discrete total="14.6 GiB" available="14.5 GiB"
time=2026-09-02T06:12:21.828Z level=INFO source=routes.go:2058 msg="vram-based default context" total_vram="29.1 GiB" default_num_ctx=32768
pulling manifest ⠋ pulling manifest ⠋ pulling manifest ⠋ 

[GIN] 2026/09/02 - 06:12:21 | 200 |      80.534µs |       127.0.0.1 | HEAD     "/"
[GIN] 2026/09/02 - 06:12:21 | 200 |     112.931µs |       127.0.0.1 | HEAD     "/"
[GIN] 2026/09/02 - 06:12:21 | 200 |      19.251µs |       127.0.0.1 | HEAD     "/"


pulling manifest ⠙ pulling manifest ⠙ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠹ pulling manifest ⠹ time=2026-09-02T06:12:22.130Z level=INFO source=download.go:181 msg="downloading 415aad607fc2 in 17 1 GB part(s)"
time=2026-09-02T06:12:22.147Z level=INFO source=download.go:181 msg="downloading ac3714bfddde in 10 100 MB part(s)"
time=2026-09-02T06:12:22.181Z level=INFO source=download.go:181 msg="downloading d372de8e9348 in 22 1 GB part(s)"
pulling manifest ⠸ pulling manifest ⠸ pulling manifest ⠸ pulling manifest 
pulling 415aad607fc2:   0% ▕                  ▏  35 MB/ 16 GB                  pulling manifest 
pulling ac3714bfddde:   1% ▕                  ▏ 8.7 MB/931 MB                  pulling manifest ⠼ pulling manifest 
pulling ac3714bfddde:   3% ▕                  ▏  27 MB/931 MB                  pulling manifest 
pulling 415aad607fc2:   0% ▕                  ▏  59 MB/ 16 GB                  pulling manifest 
pulling d372de8e9348:   0% ▕                  ▏  19 MB/ 21 G

[GIN] 2026/09/02 - 06:19:18 | 200 |         6m56s |       127.0.0.1 | POST     "/api/pull"
[GIN] 2026/09/02 - 06:19:31 | 200 |          7m9s |       127.0.0.1 | POST     "/api/pull"
[GIN] 2026/09/02 - 06:19:51 | 200 |         7m29s |       127.0.0.1 | POST     "/api/pull"


KeyboardInterrupt: 